In [ ]:
import gradio as gr
from openai import OpenAI
from kittentts import KittenTTS
from diffusers import AutoPipelineForText2Image
from tqdm.notebook import tqdm
import ipywidgets as widgets
import soundfile as sf
import numpy as np
import torch
import re
import os

In [ ]:
import torch
# Check if Intel GPU support is active
xpu_available = torch.xpu.is_available()
print(f"Is Intel GPU (XPU) available? {xpu_available}")

if xpu_available:
    print(f"Using Device: {torch.xpu.get_device_name(0)}")
else:
    print("XPU not detected. You may need to update your Intel Graphics Drivers.")

In [ ]:
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
tts_model = KittenTTS("KittenML/kitten-tts-nano-0.1")
pipe = AutoPipelineForText2Image.from_pretrained(
    "runwayml/stable-diffusion-v1-5", 
    torch_dtype=torch.float16, 
    variant="fp16",
    low_cpu_mem_usage=True
).to("cuda")

In [ ]:
def stream_text(user_input, history):
    messages = history + [{"role": "user", "content": user_input}]
    full_response = ""
    stream = client.chat.completions.create(model="llama3.2:latest", messages=messages, stream=True)
    
    for chunk in stream:
        if chunk.choices[0].delta.content:
            full_response += chunk.choices[0].delta.content
            yield messages + [{"role": "assistant", "content": full_response}], full_response

In [ ]:
# With chunking the text so that kittenTTS context window is not exceeded.
def generate_audio(text):
    if not text or len(text.strip()) == 0:
        return None
    
    audio_path = "response_audio.wav"
    MAX_CHARS = 180  # The "safety limit" for KittenTTS Nano's internal buffer
    
    # 1. Initial split by sentence markers (. ! ?)
    sentences = re.split(r'(?<=[.!?]) +', text.replace('\n', ' '))
    
    # 2. Sub-chunking: If a sentence is too long, break it by commas or spaces
    final_chunks = []
    for s in sentences:
        if len(s) <= MAX_CHARS:
            final_chunks.append(s)
        else:
            # Sub-split long sentences by commas or just chunks of words
            words = s.split(' ')
            current_chunk = ""
            for word in words:
                if len(current_chunk) + len(word) < MAX_CHARS:
                    current_chunk += " " + word
                else:
                    final_chunks.append(current_chunk.strip())
                    current_chunk = word
            final_chunks.append(current_chunk.strip())

    combined_audio = []
    
    try:
        for chunk in final_chunks:
            clean_chunk = chunk.strip()
            if len(clean_chunk) < 2:
                continue
    
            chunk_data = tts_model.generate(
                clean_chunk, 
                voice="expr-voice-2-f", 
                speed=1.0
            )
            combined_audio.append(chunk_data)
        
        if not combined_audio:
            return None

        # Stitch the clean chunks together
        final_audio = np.concatenate(combined_audio)
        sf.write(audio_path, final_audio, 24000)
        return audio_path

    except Exception as e:
        print(f"Modular TTS Error: {e}")
        return None

In [ ]:
def extract_keyword_and_craft_image_prompt(full_text):
    if not full_text: 
        return "a cute simple cat, flat colors, cartoon style"
    
    # Few-shot examples to guide the model's style and logic
    system_prompt = '''
        You are an AI that converts long chat responses into simple, cartoonish image prompts. 
        Your goal is to identify the main subject and describe it in a minimalist, 2D vector art style. 
        Avoid words like '8k', 'realistic', 'detailed', or 'photorealistic'. Use 'flat colors' and 'simple lines'.\n\n
        ### FEW-SHOT EXAMPLES ###\n
        Input: 'In the heart of the enchanted forest, a small blue owl with glowing eyes sat on a gnarled oak branch, watching the fireflies dance.'\n
        Output: A cute blue owl on a tree branch, simple cartoon style, flat colors, friendly fireflies background\n\n
        Input: 'The futuristic robot hovered over the neon-lit streets of Tokyo, its metallic surface reflecting the vibrant pink and blue signs.'\n
        Output: A friendly hovering robot, simple 2D vector art, bright neon colors, minimalist city background\n\n
        Input: 'A weary traveler reached the peak of the snowy mountain just as the golden sun began to set over the jagged horizon.'\n
        Output: A cute hiker on a snowy mountain top, simple cartoon style, orange sunset sky, flat illustration\n
        ### END EXAMPLES ###
    '''
    
    messages2 = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Convert this text to a image generation prompt: {full_text}"}
    ]
    
    try:
        response = client.chat.completions.create(
            model="llama3.2:latest",
            messages=messages2,  # type: ignore
            stream=False,
        )
                                       
        enhanced_prompt = response.choices[0].message.content.strip()
        # Cleaning up any unintended quotes or prefixes
        enhanced_prompt = re.sub(r'^(Prompt:|Output:)\s*', '', enhanced_prompt, flags=re.IGNORECASE)
        print(f"DEBUG: Cartoon Prompt -> {enhanced_prompt}")
        return enhanced_prompt
    except Exception as e:
        print(f"Prompt Crafting Error: {e}")
        return "a simple cute character, cartoon style"

In [ ]:
def generate_image(text):
    if not text: 
        return None
    
    try:
        image = pipe(
            prompt=text, 
            num_inference_steps=20, 
            guidance_scale=7.5
        ).images[0]
        
        image_path = "generated_image.png"
        image.save(image_path)
        return image_path
    except Exception as e:
        print(f"Image Gen Error: {e}")
        return None

In [ ]:
# --- THE CORRECTED UI LAYOUT ---
with gr.Blocks(fill_height=True, title="Modular AI Studio") as demo:
    # This 'State' acts as a hidden bridge between the Text function and Audio function
    final_text_state = gr.State("")

    gr.HTML("<h1 style='text-align: center; font-family: sans-serif;'>AI Multimodal Studio</h1>")
    
    with gr.Row(equal_height=True):
        # LEFT: Chat Column
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(label="Conversation", height=600)
            with gr.Row():
                msg = gr.Textbox(show_label=False, placeholder="Ask me...", scale=9, container=False)
                submit_btn = gr.Button("Send", variant="primary", scale=1)

        # RIGHT: Media Column
        with gr.Column(scale=1):
            image_viewer = gr.Image(label="Image Output", height=400, interactive=False)
            audio_player = gr.Audio(label="Voice Output", interactive=False, autoplay=True)
    
    # STEP 1: When user hits 'Send', run stream_text
    # It updates the chatbot (stream) and the final_text_state (at the very end)
    text_event = submit_btn.click(
        fn=stream_text, 
        inputs=[msg, chatbot], 
        outputs=[chatbot, final_text_state]
    )
    
    submit_btn.click(lambda: "", None, [msg])

    # STEP 2: When stream_text finishes, trigger generate_audio
    # It takes the 'final_text_state' as input
    text_event.then(
        fn=generate_audio, 
        inputs=[final_text_state], 
        outputs=[audio_player]
    )
    
    # Intermediate state to hold the 'processed' image prompt
    image_prompt_state = gr.State("")

    # STEP 2.5: After text is finished, find the keyword/craft the prompt
    text_event.then(
        fn=extract_keyword_and_craft_image_prompt,
        inputs=[final_text_state],
        outputs=[image_prompt_state] # Save the refined prompt here
    )

    # STEP 3: Trigger generate_image using the REFINED prompt
    text_event.then(
        fn=generate_image, 
        inputs=[image_prompt_state], # Use the crafted prompt instead of raw text
        outputs=[image_viewer]
    )

    # STEP 4: Clear the input box after everything starts
    text_event.then(lambda: "", None, [msg])

    # Repeat for the 'Enter' key on the Textbox
    submit_event = msg.submit(
        fn=stream_text, 
        inputs=[msg, chatbot], 
        outputs=[chatbot, final_text_state]
    )
    msg.submit(lambda: "", None, [msg])

    submit_event.then(
        fn=generate_audio, 
        inputs=[final_text_state], 
        outputs=[audio_player]
    )
    submit_event.then(
        fn=extract_keyword_and_craft_image_prompt, # Add this
        inputs=[final_text_state],
        outputs=[image_prompt_state]
    )
    submit_event.then(
        fn=generate_image, 
        inputs=[image_prompt_state], # Use the refined prompt state
        outputs=[image_viewer]
    )
    submit_event.then(lambda: "", None, [msg])

if __name__ == "__main__":
    demo.launch()